# Steganography Model Robustness Testing

This notebook tests the robustness of a trained steganography model by:
1. Loading a trained model
2. Encoding a secret message into a cover image
3. Applying various distortions (attacks) to the stego image
4. Measuring recovery accuracy (BER) and image quality (PSNR, SSIM)
5. Visualizing results

## Metrics:
- **BER (Bit Error Rate)**: Measures message recovery accuracy (lower is better)
- **PSNR (Peak Signal-to-Noise Ratio)**: Measures image quality in dB (higher is better, >30 dB is good)
- **SSIM (Structural Similarity)**: Measures perceptual similarity (higher is better, >0.9 is good)

## 1. Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import steganography modules
from models.model import StegoModel
from attacks import (
    JPEGCompression, GaussianNoise, ResizeAttack, 
    GaussianBlur, RandomCrop, ColorJitter
)
from evaluation import compute_ber, compute_psnr, compute_ssim

# Setup matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 2. Configuration and Setup

In [ ]:
# Configuration
CONFIG = {
    'model_path': 'checkpoints/best_model.pth',  # Path to trained model
    'image_size': 256,
    'message_length': 1024,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'test_message': 'This is a secret message for robustness testing!'
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Load Trained Model

Load the pre-trained steganography model from checkpoint.

In [ ]:
def load_model(model_path, message_length, device):
    """Load trained steganography model."""
    model = StegoModel(message_length=message_length)
    
    # Load checkpoint
    if Path(model_path).exists():
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"✓ Model loaded from checkpoint (epoch {checkpoint.get('epoch', 'unknown')})")
        else:
            model.load_state_dict(checkpoint)
            print(f"✓ Model loaded from state dict")
    else:
        print(f"⚠ Warning: Model file not found at {model_path}")
        print(f"  Using randomly initialized model for demonstration")
    
    model = model.to(device)
    model.eval()
    
    return model

# Load model
model = load_model(
    CONFIG['model_path'], 
    CONFIG['message_length'], 
    CONFIG['device']
)

print(f"\nModel architecture:")
print(f"  Encoder parameters: {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"  Decoder parameters: {sum(p.numel() for p in model.decoder.parameters()):,}")

## 4. Prepare Test Image and Message

Create a synthetic test image or load your own image.

In [ ]:
def create_test_image(size=256):
    """Create a colorful test image with gradients and patterns."""
    # Create RGB channels with different patterns
    x = np.linspace(0, 1, size)
    y = np.linspace(0, 1, size)
    X, Y = np.meshgrid(x, y)
    
    # Red channel: diagonal gradient
    R = (X + Y) / 2
    
    # Green channel: circular pattern
    center_x, center_y = 0.5, 0.5
    G = 1 - np.sqrt((X - center_x)**2 + (Y - center_y)**2) / np.sqrt(2)
    
    # Blue channel: wave pattern
    B = (np.sin(X * 10) + np.sin(Y * 10)) / 2 + 0.5
    
    # Stack channels
    image = np.stack([R, G, B], axis=2)
    image = np.clip(image, 0, 1).astype(np.float32)
    
    return torch.from_numpy(image).permute(2, 0, 1).unsqueeze(0)

def text_to_binary_message(text, message_length):
    """Convert text to binary message tensor."""
    # Encode text to bytes
    text_bytes = text.encode('utf-8')
    binary_str = ''.join(format(byte, '08b') for byte in text_bytes)
    
    # Add length prefix
    length_bits = format(len(binary_str), '032b')
    binary_str = length_bits + binary_str
    
    # Pad or truncate
    if len(binary_str) < message_length:
        binary_str += '0' * (message_length - len(binary_str))
    else:
        binary_str = binary_str[:message_length]
    
    # Convert to tensor
    binary_list = [float(bit) for bit in binary_str]
    return torch.tensor([binary_list], dtype=torch.float32)

# Create test data
cover_image = create_test_image(CONFIG['image_size']).to(CONFIG['device'])
message = text_to_binary_message(CONFIG['test_message'], CONFIG['message_length']).to(CONFIG['device'])

print(f"✓ Test image created: {cover_image.shape}")
print(f"✓ Message prepared: {CONFIG['test_message']}")
print(f"  Message length: {CONFIG['message_length']} bits")

# Visualize cover image
plt.figure(figsize=(6, 6))
plt.imshow(cover_image.squeeze(0).permute(1, 2, 0).cpu().numpy())
plt.title('Cover Image', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## 5. Encode Message into Cover Image

Create the stego image by hiding the secret message.

In [ ]:
# Encode message
with torch.no_grad():
    stego_image = model.encode(cover_image, message)
    
    # Verify encoding quality
    psnr_stego = compute_psnr(stego_image, cover_image)
    ssim_stego = compute_ssim(stego_image, cover_image)
    
    # Test decoding from clean stego
    decoded_logits = model.decode(stego_image)
    ber_clean = compute_ber(decoded_logits, message, logits=True)

print("✓ Message encoded successfully")
print(f"\nStego Image Quality:")
print(f"  PSNR: {psnr_stego:.2f} dB")
print(f"  SSIM: {ssim_stego:.4f}")
print(f"  BER (no attack): {ber_clean:.6f} ({(1-ber_clean)*100:.2f}% accuracy)")

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(cover_image.squeeze(0).permute(1, 2, 0).cpu().numpy())
axes[0].set_title('Cover Image', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(stego_image.squeeze(0).permute(1, 2, 0).cpu().numpy())
axes[1].set_title(f'Stego Image\nPSNR: {psnr_stego:.2f} dB, SSIM: {ssim_stego:.4f}', 
                  fontsize=12, fontweight='bold')
axes[1].axis('off')

# Difference (amplified)
diff = torch.abs(stego_image - cover_image).squeeze(0).permute(1, 2, 0).cpu().numpy()
diff_amplified = np.clip(diff * 10, 0, 1)  # Amplify differences
axes[2].imshow(diff_amplified)
axes[2].set_title('Difference (10x amplified)', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 6. Define Attack/Distortion Functions

Set up various attacks to test model robustness.

In [ ]:
# Define attacks with various parameters
attacks = {
    'No Attack': lambda x: x,
    'JPEG Q=90': JPEGCompression(quality=90),
    'JPEG Q=75': JPEGCompression(quality=75),
    'JPEG Q=50': JPEGCompression(quality=50),
    'Gaussian Noise (σ=0.01)': GaussianNoise(std=0.01),
    'Gaussian Noise (σ=0.05)': GaussianNoise(std=0.05),
    'Resize 0.75x': ResizeAttack(scale_factor=0.75),
    'Resize 0.5x': ResizeAttack(scale_factor=0.5),
    'Gaussian Blur (σ=1.0)': GaussianBlur(kernel_size=5, sigma=1.0),
    'Gaussian Blur (σ=2.0)': GaussianBlur(kernel_size=7, sigma=2.0),
    'Random Crop 90%': RandomCrop(crop_ratio=0.9),
    'Random Crop 80%': RandomCrop(crop_ratio=0.8),
    'Color Jitter': ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
}

print(f"✓ Defined {len(attacks)} attack scenarios:")
for i, name in enumerate(attacks.keys(), 1):
    print(f"  {i:2d}. {name}")

## 7. Apply Attacks and Measure Metrics

Test model robustness by applying each attack and measuring BER, PSNR, and SSIM.

In [ ]:
# Test all attacks
results = []

print("Testing robustness against attacks...")
print("=" * 70)

with torch.no_grad():
    for attack_name, attack_fn in attacks.items():
        print(f"\nTesting: {attack_name}")
        
        # Apply attack
        attacked_image = attack_fn(stego_image)
        
        # Decode message
        decoded_logits = model.decode(attacked_image)
        
        # Compute metrics
        ber = compute_ber(decoded_logits, message, logits=True)
        psnr = compute_psnr(attacked_image, stego_image)
        ssim = compute_ssim(attacked_image, stego_image)
        
        # Store results
        results.append({
            'Attack': attack_name,
            'BER': ber,
            'Accuracy (%)': (1 - ber) * 100,
            'PSNR (dB)': psnr,
            'SSIM': ssim
        })
        
        print(f"  BER: {ber:.6f} | Accuracy: {(1-ber)*100:.2f}% | PSNR: {psnr:.2f} dB | SSIM: {ssim:.4f}")

print("\n" + "=" * 70)
print("✓ Robustness testing completed")

# Create results DataFrame
results_df = pd.DataFrame(results)
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
print(results_df.to_string(index=False))
print("=" * 70)

## 8. Visualize Results

Create comprehensive visualizations of the robustness test results.

In [ ]:
# Plot 1: Metrics Bar Charts
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# BER (lower is better)
axes[0].barh(results_df['Attack'], results_df['BER'], color='coral')
axes[0].set_xlabel('Bit Error Rate', fontsize=12, fontweight='bold')
axes[0].set_title('BER (Lower is Better)', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
axes[0].invert_yaxis()

# PSNR (higher is better)
axes[1].barh(results_df['Attack'], results_df['PSNR (dB)'], color='skyblue')
axes[1].set_xlabel('PSNR (dB)', fontsize=12, fontweight='bold')
axes[1].set_title('PSNR (Higher is Better)', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
axes[1].invert_yaxis()

# SSIM (higher is better)
axes[2].barh(results_df['Attack'], results_df['SSIM'], color='lightgreen')
axes[2].set_xlabel('SSIM', fontsize=12, fontweight='bold')
axes[2].set_title('SSIM (Higher is Better)', fontsize=14, fontweight='bold')
axes[2].grid(axis='x', alpha=0.3)
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Accuracy percentage
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['green' if acc > 95 else 'orange' if acc > 90 else 'red' 
          for acc in results_df['Accuracy (%)']]

bars = ax.barh(results_df['Attack'], results_df['Accuracy (%)'], color=colors, alpha=0.7)

ax.set_xlabel('Message Recovery Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Message Recovery Accuracy Under Different Attacks', fontsize=14, fontweight='bold')
ax.set_xlim(0, 105)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

# Add value labels
for i, (attack, acc) in enumerate(zip(results_df['Attack'], results_df['Accuracy (%)'])):
    ax.text(acc + 1, i, f'{acc:.1f}%', va='center', fontsize=9)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', alpha=0.7, label='Excellent (>95%)'),
    Patch(facecolor='orange', alpha=0.7, label='Good (90-95%)'),
    Patch(facecolor='red', alpha=0.7, label='Poor (<90%)')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## 9. Visual Comparison: Original vs Stego vs Attacked Images

Display selected attacks side-by-side for visual inspection.

In [ ]:
# Select interesting attacks to visualize
selected_attacks = [
    'No Attack',
    'JPEG Q=50',
    'Gaussian Noise (σ=0.05)',
    'Resize 0.5x',
    'Gaussian Blur (σ=2.0)',
    'Random Crop 80%'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

with torch.no_grad():
    for idx, attack_name in enumerate(selected_attacks):
        if attack_name in attacks:
            # Apply attack
            attack_fn = attacks[attack_name]
            attacked_image = attack_fn(stego_image)
            
            # Get metrics
            result = results_df[results_df['Attack'] == attack_name].iloc[0]
            
            # Display
            axes[idx].imshow(attacked_image.squeeze(0).permute(1, 2, 0).cpu().numpy())
            axes[idx].set_title(
                f'{attack_name}\n'
                f'BER: {result["BER"]:.4f} | Acc: {result["Accuracy (%)"]:.1f}%\n'
                f'PSNR: {result["PSNR (dB)"]:.1f} dB | SSIM: {result["SSIM"]:.3f}',
                fontsize=10,
                fontweight='bold'
            )
            axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 10. Summary Statistics

Analyze overall robustness performance.

In [ ]:
# Exclude 'No Attack' for statistics
attack_results = results_df[results_df['Attack'] != 'No Attack']

print("=" * 70)
print("ROBUSTNESS SUMMARY STATISTICS")
print("=" * 70)
print()
print("Bit Error Rate (BER):")
print(f"  Mean:   {attack_results['BER'].mean():.6f}")
print(f"  Median: {attack_results['BER'].median():.6f}")
print(f"  Min:    {attack_results['BER'].min():.6f}")
print(f"  Max:    {attack_results['BER'].max():.6f}")
print()
print("Message Recovery Accuracy:")
print(f"  Mean:   {attack_results['Accuracy (%)'].mean():.2f}%")
print(f"  Median: {attack_results['Accuracy (%)'].median():.2f}%")
print(f"  Min:    {attack_results['Accuracy (%)'].min():.2f}%")
print(f"  Max:    {attack_results['Accuracy (%)'].max():.2f}%")
print()
print("PSNR (dB):")
print(f"  Mean:   {attack_results['PSNR (dB)'].mean():.2f}")
print(f"  Median: {attack_results['PSNR (dB)'].median():.2f}")
print(f"  Min:    {attack_results['PSNR (dB)'].min():.2f}")
print(f"  Max:    {attack_results['PSNR (dB)'].max():.2f}")
print()
print("SSIM:")
print(f"  Mean:   {attack_results['SSIM'].mean():.4f}")
print(f"  Median: {attack_results['SSIM'].median():.4f}")
print(f"  Min:    {attack_results['SSIM'].min():.4f}")
print(f"  Max:    {attack_results['SSIM'].max():.4f}")
print()
print("=" * 70)

# Count performance categories
excellent = (attack_results['Accuracy (%)'] > 95).sum()
good = ((attack_results['Accuracy (%)'] >= 90) & (attack_results['Accuracy (%)'] <= 95)).sum()
poor = (attack_results['Accuracy (%)'] < 90).sum()

print("\nPerformance Categories:")
print(f"  Excellent (>95% accuracy): {excellent}/{len(attack_results)}")
print(f"  Good (90-95% accuracy):    {good}/{len(attack_results)}")
print(f"  Poor (<90% accuracy):      {poor}/{len(attack_results)}")
print("=" * 70)

## 11. Conclusion

### Key Findings:

The robustness test evaluates how well the steganography model preserves hidden messages under various image distortions. 

**What the metrics mean:**
- **BER (Bit Error Rate)**: Percentage of incorrectly decoded bits. Lower is better (0.0 = perfect recovery).
- **Accuracy**: Percentage of correctly decoded bits. Higher is better (100% = perfect recovery).
- **PSNR**: Image quality in decibels. Higher is better (>30 dB = good quality, >40 dB = excellent).
- **SSIM**: Structural similarity. Higher is better (>0.9 = excellent, 1.0 = identical).

**Typical performance expectations:**
- **Excellent robustness**: >95% accuracy under common attacks (JPEG compression, mild noise)
- **Good robustness**: 90-95% accuracy under moderate attacks
- **Poor robustness**: <90% accuracy indicates vulnerability to that attack type

**Next steps:**
1. If robustness is insufficient, consider training with adversarial augmentation
2. Add defense modules (anti-aliasing, denoising) before decoding
3. Test with your own images using the code cells above
4. Adjust attack parameters to test edge cases

---

### How to use this notebook with your own data:

1. **Replace model**: Change `CONFIG['model_path']` to your trained model
2. **Load your image**: Replace the `create_test_image()` call with:
   ```python
   from torchvision import transforms
   image = Image.open('your_image.jpg').convert('RGB')
   transform = transforms.Compose([
       transforms.Resize((256, 256)),
       transforms.ToTensor()
   ])
   cover_image = transform(image).unsqueeze(0).to(CONFIG['device'])
   ```
3. **Customize message**: Change `CONFIG['test_message']`
4. **Add attacks**: Add entries to the `attacks` dictionary
5. **Run all cells** to see results